<a href="https://colab.research.google.com/github/2403a54123-web/NLP/blob/main/2403A54123_NLP_Lab_Assignment_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import re

In [7]:
df = pd.read_csv("spam.csv")
print(df.head())
print(df['label'].value_counts())

  label                                   text
0   ham          Hey, how are you doing today?
1  spam              Win a free lottery now!!!
2   ham                     Let's meet at 5 pm
3  spam  Congratulations! You have won a prize
4   ham              Call me when you are free
label
ham     100
spam    100
Name: count, dtype: int64


In [8]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['text'] = df['text'].apply(clean_text)
print(df.head())

  label                                  text
0   ham           hey how are you doing today
1  spam                win a free lottery now
2   ham                      lets meet at  pm
3  spam  congratulations you have won a prize
4   ham             call me when you are free


In [9]:
from collections import Counter

all_words = []

for sentence in df['text']:
    all_words.extend(sentence.split())

vocab = {word: i+1 for i, (word, _) in enumerate(Counter(all_words).items())}

print("Vocabulary size:", len(vocab))

Vocabulary size: 39


In [10]:
def encode(text):
    return [vocab.get(word, 0) for word in text.split()]

df['encoded'] = df['text'].apply(encode)
print(df.head())

  label                                  text                 encoded
0   ham           hey how are you doing today      [1, 2, 3, 4, 5, 6]
1  spam                win a free lottery now       [7, 8, 9, 10, 11]
2   ham                      lets meet at  pm        [12, 13, 14, 15]
3  spam  congratulations you have won a prize  [16, 4, 17, 18, 8, 19]
4   ham             call me when you are free   [20, 21, 22, 4, 3, 9]


In [11]:
max_len = 20

def pad(seq):
    return seq[:max_len] + [0]*(max_len - len(seq))

df['padded'] = df['encoded'].apply(pad)

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.array(df['padded'].tolist())
y = np.array([1 if label == 'spam' else 0 for label in df['label']])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 160
Test size: 40


In [13]:
embedding_dim = 50

embedding_matrix = np.random.rand(len(vocab)+1, embedding_dim)

In [14]:
import torch
import torch.nn as nn

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(TextCNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.conv = nn.Conv1d(embedding_dim, 64, kernel_size=3)
        self.pool = nn.MaxPool1d(2)

        self.fc = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)

        x = torch.relu(self.conv(x))
        x = self.pool(x)

        x = torch.mean(x, dim=2)
        x = self.fc(x)

        return self.sigmoid(x)

In [15]:
model = TextCNN(len(vocab)+1, embedding_dim)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

X_train_t = torch.tensor(X_train, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

for epoch in range(5):
    outputs = model(X_train_t).squeeze()
    loss = criterion(outputs, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.717298150062561
Epoch 2, Loss: 0.6900875568389893
Epoch 3, Loss: 0.6760373115539551
Epoch 4, Loss: 0.669302761554718
Epoch 5, Loss: 0.6636492013931274


In [16]:
from sklearn.metrics import classification_report, confusion_matrix

X_test_t = torch.tensor(X_test, dtype=torch.long)

preds = model(X_test_t).detach().numpy()
pred_labels = (preds > 0.5).astype(int)

print(classification_report(y_test, pred_labels))
print(confusion_matrix(y_test, pred_labels))

              precision    recall  f1-score   support

           0       0.55      1.00      0.71        22
           1       0.00      0.00      0.00        18

    accuracy                           0.55        40
   macro avg       0.28      0.50      0.35        40
weighted avg       0.30      0.55      0.39        40

[[22  0]
 [18  0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
